In [1]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade",
    "pyarrow", "pandas", "scikit-learn", "umap-learn", "matplotlib", "seaborn",
    "huggingface_hub", "tqdm"], check=True)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 90.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 625.2/625.2 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 67.1 MB/s eta 0:00:00


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
ydata-profiling 4.18.1 requires matplotlib<=3.10,>=3.5, but you have matplotlib 3.10.8 which is incompatible.
ydata-profiling 4.18.1 requires pandas!=1.4.0,<3.0,>1.5, but you have pandas 3.0.1 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.1 which is incompatible.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
db-dtypes 1.5.0 requires pandas<3.0.0,>=1.5.3, but you have pandas 3.0.1 which is incompatible.
bqplot 0.12.45 requires pandas<3.0.0,>=1.0.0, but you have pandas 3.0.1 which is incompatible.
gradio 5.50.0 requires pandas<3.0

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '--upgrade', 'pyarrow', 'pandas', 'scikit-learn', 'umap-learn', 'matplotlib', 'seaborn', 'huggingface_hub', 'tqdm'], returncode=0)

In [ ]:
import os, shutil
import pandas as pd
import numpy as np
from glob import glob
from tqdm import tqdm
from huggingface_hub import hf_hub_download

REPO_ID = "Tungtom2004/Telegram_politic_dataset"
CHANNEL_RANGE = range(10, 25)

rows = []
errors = []

for i in CHANNEL_RANGE:
    zip_name = f"channels_{i}_parquet.zip"
    raw_dir = f"./channels_{i}_parquet"

    print(f"\n{'='*50}")
    print(f"Processing channels_{i}")

    try:
        local_zip = hf_hub_download(
            repo_id=REPO_ID, filename=zip_name,
            repo_type="dataset", revision="main"
        )
        os.system(f"unzip -q '{local_zip}' -d .")
    except Exception as e:
        print(f"FAIL download: {e}")
        continue

    parquet_files = sorted(glob(f"{raw_dir}/*.parquet"))
    print(f"  {len(parquet_files)} files")

    for pf in tqdm(parquet_files, desc=f"ch_{i}", leave=False):
        try:
            feat = extract_features(pf)
            if feat is not None:
                rows.append(feat)
        except Exception as e:
            errors.append((os.path.basename(pf), str(e)))

    shutil.rmtree(raw_dir, ignore_errors=True)

    cache = os.path.expanduser("~/.cache/huggingface")
    if os.path.exists(cache):
        shutil.rmtree(cache, ignore_errors=True)

    print(f"  Done. Channels so far: {len(rows)} | Errors: {len(errors)}")

df_profiles = pd.DataFrame(rows)
print(f"\nTotal channels: {len(df_profiles)}")
print(f"Total errors: {len(errors)}")


Processing channels_10


channels_10_parquet.zip:   0%|          | 0.00/1.93G [00:00<?, ?B/s]

  0 files


  Done. Channels so far: 0 | Errors: 0

Processing channels_11


channels_11_parquet.zip:   0%|          | 0.00/6.08G [00:00<?, ?B/s]

...

channels_13_parquet.zip:   0%|          | 0.00/7.96G [00:00<?, ?B/s]

./kaggle/working/channels_13_parquet/channel_1306192062.parquet:  write error (disk full?).  Continue? (y/n/^C) 


  0 files


  Done. Channels so far: 0 | Errors: 0

Processing channels_14


channels_14_parquet.zip:   0%|          | 0.00/8.93G [00:00<?, ?B/s]

In [ ]:
import pandas as pd
import numpy as np

def extract_features(parquet_path):
    df = pd.read_parquet(parquet_path)
    if len(df) == 0:
        return None

    channel_name = df["channel_name"].iloc[0] if "channel_name" in df.columns else os.path.basename(parquet_path).replace(".parquet", "")

    df["_ts"] = pd.to_datetime(df["date"], errors="coerce")
    df = df[df["_ts"].notna()].copy()
    if len(df) < 5:
        return None

    df["_hour"] = df["_ts"].dt.hour
    df["_dow"] = df["_ts"].dt.dayofweek
    df["_date"] = df["_ts"].dt.date

    n = len(df)
    active_days = df["_date"].nunique()
    date_span = max((df["_ts"].max() - df["_ts"].min()).days, 1)

    daily_counts = df.groupby("_date").size()
    burst = daily_counts.std() / daily_counts.mean() if daily_counts.mean() > 0 else 0

    content_lens = df["content"].fillna("").str.strip().str.split().str.len().fillna(0)

    frac = lambda cond: cond.sum() / n

    features = {
        "channel_name": channel_name,

        "total_messages": n,
        "active_days": active_days,
        "msgs_per_day": n / date_span,
        "burstiness": burst,
        "frac_night": frac((df["_hour"] >= 0) & (df["_hour"] < 6)),
        "frac_morning": frac((df["_hour"] >= 6) & (df["_hour"] < 12)),
        "frac_afternoon": frac((df["_hour"] >= 12) & (df["_hour"] < 18)),
        "frac_evening": frac((df["_hour"] >= 18) & (df["_hour"] < 24)),
        "frac_weekend": frac(df["_dow"].isin([5, 6])),

        "avg_views": df["views"].mean(),
        "median_views": df["views"].median(),
        "max_views": df["views"].max(),
        "avg_forwards": df["forwards"].mean(),
        "forward_ratio": frac(df["forwards"] > 0),
        "avg_replies": df["replies"].mean(),
        "reply_ratio": frac(df["replies"] > 0),
        "frac_is_forward": frac(df["forward"] > 0),
        "fwd_per_view": df["forwards"].mean() / max(df["views"].mean(), 1),
        "reply_per_view": df["replies"].mean() / max(df["views"].mean(), 1),

        "avg_word_count": content_lens.mean(),
        "frac_photo": frac(df["photo"] > 0),
        "frac_video": frac(df["video"] > 0),
        "frac_voice": frac(df["voice"] > 0),
        "frac_web_preview": frac(df["web_preview"] > 0),
        "frac_buttons": frac(df["buttons"] > 0),
        "frac_text_only": frac(
            (df["photo"] == 0) & (df["video"] == 0) &
            (df["voice"] == 0) & (df["other_media"] == 0) &
            (df["content"].fillna("").str.strip().str.len() > 0)
        ),
        "frac_has_urls": frac(df["urls"].fillna("").str.len() > 2),
        "frac_has_hashtags": frac(df["hashtags"].fillna("").str.len() > 2),

        "avg_toxicity": df["toxicity"].mean(),
        "avg_severe_toxicity": df["severe_toxicity"].mean(),
        "avg_identity_attack": df["identity_attack"].mean(),
        "avg_insult": df["insult"].mean(),
        "avg_profanity": df["profanity"].mean(),
        "avg_threat": df["threat"].mean(),
        "avg_political": pd.to_numeric(df["political"], errors="coerce").mean(),
        "frac_high_toxic": frac(df["toxicity"] > 0.5),

        "n_languages": df["language"].nunique(),
        "n_fwd_sources": df["chain_from_id"].fillna("").str.strip().replace("", pd.NA).dropna().nunique(),
        "frac_chain": frac(df["chain_from_id"].fillna("").str.strip().str.len() > 0),
    }

    return features

In [ ]:
from glob import glob
from tqdm import tqdm

all_parquets = []
for i in CHANNEL_RANGE:
    all_parquets += sorted(glob(f"./channels_{i}_parquet/*.parquet"))

print(f"Total parquet files: {len(all_parquets)}")

rows = []
errors = []

for pf in tqdm(all_parquets, desc="Extracting features"):
    try:
        feat = extract_features(pf)
        if feat is not None:
            rows.append(feat)
    except Exception as e:
        errors.append((pf, str(e)))

df_profiles = pd.DataFrame(rows)

print(f"Channels with features: {len(df_profiles)}")
print(f"Errors: {len(errors)}")
if errors:
    for path, err in errors[:5]:
        print(f"  {os.path.basename(path)}: {err}")

In [ ]:
df_profiles = df_profiles[
    (df_profiles["total_messages"] >= 10) &
    (df_profiles["active_days"] >= 3)
].copy()

df_profiles = df_profiles.fillna(0)
df_profiles.to_csv("channel_profiles.csv", index=False)

print(f"Final: {len(df_profiles)} channels")
print(f"Columns: {len(df_profiles.columns)}")
df_profiles.describe().round(2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

axes[0,0].hist(np.log10(df_profiles["total_messages"].clip(1)), bins=50)
axes[0,0].set_title("log10(total_messages)")

axes[0,1].hist(df_profiles["avg_toxicity"], bins=50)
axes[0,1].set_title("avg_toxicity")

axes[0,2].hist(df_profiles["frac_is_forward"], bins=50)
axes[0,2].set_title("frac_is_forward")

axes[1,0].hist(np.log10(df_profiles["avg_views"].clip(1)), bins=50)
axes[1,0].set_title("log10(avg_views)")

axes[1,1].hist(df_profiles["msgs_per_day"], bins=50)
axes[1,1].set_title("msgs_per_day")

axes[1,2].hist(df_profiles["burstiness"], bins=50)
axes[1,2].set_title("burstiness")

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.preprocessing import RobustScaler

feature_cols = [c for c in df_profiles.columns if c != "channel_name"]

X = df_profiles[feature_cols].values

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X)

print(f"Feature matrix: {X_scaled.shape}")

In [ ]:
from sklearn.cluster import KMeans

inertias = []
K_range = range(2, 16)

for k in K_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=42, max_iter=300)
    km.fit(X_scaled)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 4))
plt.plot(list(K_range), inertias, "o-")
plt.xlabel("K")
plt.ylabel("Inertia")
plt.title("Elbow method")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
K = 5

km = KMeans(n_clusters=K, n_init=20, random_state=42, max_iter=500)
df_profiles["cluster"] = km.fit_predict(X_scaled)

print(df_profiles["cluster"].value_counts().sort_index())

In [ ]:
import umap

reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=30, min_dist=0.3)
embedding = reducer.fit_transform(X_scaled)

df_profiles["umap_x"] = embedding[:, 0]
df_profiles["umap_y"] = embedding[:, 1]

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    df_profiles["umap_x"], df_profiles["umap_y"],
    c=df_profiles["cluster"], cmap="tab10", s=8, alpha=0.6
)
plt.colorbar(scatter, label="Cluster")
plt.title("Channel archetypes (UMAP)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.show()

In [ ]:
interpret_cols = [
    "total_messages", "msgs_per_day", "burstiness",
    "avg_views", "avg_forwards", "forward_ratio", "reply_ratio",
    "frac_is_forward", "frac_chain",
    "avg_toxicity", "frac_high_toxic", "avg_political",
    "frac_photo", "frac_video", "frac_text_only",
    "frac_has_urls", "avg_word_count", "n_fwd_sources",
]

cluster_means = df_profiles.groupby("cluster")[interpret_cols].mean().round(3)
print(cluster_means.T.to_string())

In [ ]:
df_profiles.to_csv("channel_profiles_clustered.csv", index=False)
cluster_means.T.to_csv("cluster_centroids.csv")

print("Saved: channel_profiles_clustered.csv")
print("Saved: cluster_centroids.csv")